# Random_Forest — teste final walk-forward

Parâmetros congelados, mesmas origens dos quatro modelos, horizonte de 1 hora e reajuste em **toda origem**. Esta é uma execução pesada.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
params = load_json(require_file(
    config.RESULT_DIR / "best_params_Random_Forest.json",
    "01_otimizacao_hiperparametros.ipynb",
))
origins = models.final_origins(model_df)
print("Parâmetros:", params)
print("Origens finais:", len(origins))
res = models.evaluate(model_df, origins, "Random_Forest", params)

models.save_result(res, config.RESULT_DIR)

# Diagnóstico residual em 168 horas consecutivas. Essas previsões não alteram
# o MAE oficial nem a seleção; servem para ACF e Ljung-Box com espaçamento 1h.
diagnostic_origins = models.residual_diagnostic_origins(model_df)
diagnostic = models.evaluate(
    model_df, diagnostic_origins, "Random_Forest", params, progress_every=24
)
diagnostic.predictions.to_csv(
    config.RESULT_DIR / "residual_diagnostic_Random_Forest.csv", index=False
)

baseline_path = config.RESULT_DIR / "baseline_predictions.csv"
baseline = models.baseline_predictions(model_df, origins)
baseline.to_csv(baseline_path, index=False)

print(f"MAE final = {res.mae:.3f}")
print(f"Tempo = {res.runtime_sec / 60:.1f} min")
print(f"Diagnóstico residual consecutivo = {diagnostic.runtime_sec / 60:.1f} min")
display(res.predictions.head())
display(res.predictions.tail())


Parâmetros: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 900}
Origens finais: 521
Random_Forest: 10/521 origens | MAE parcial=302.194
Random_Forest: 20/521 origens | MAE parcial=229.725
Random_Forest: 30/521 origens | MAE parcial=185.626
Random_Forest: 40/521 origens | MAE parcial=167.389
Random_Forest: 50/521 origens | MAE parcial=182.016
Random_Forest: 60/521 origens | MAE parcial=171.405
Random_Forest: 70/521 origens | MAE parcial=183.366
Random_Forest: 80/521 origens | MAE parcial=173.317
Random_Forest: 90/521 origens | MAE parcial=180.583
Random_Forest: 100/521 origens | MAE parcial=178.042
Random_Forest: 110/521 origens | MAE parcial=188.619
Random_Forest: 120/521 origens | MAE parcial=183.282
Random_Forest: 130/521 origens | MAE parcial=179.784
Random_Forest: 140/521 origens | MAE parcial=174.899
Random_Forest: 150/521 origens | MAE parcial=173.742
Random_Forest: 160/521 origens | MAE parcial=184.958
Random_Forest: 17

,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
0,22908,2018-02-04 07:00:00,2018-02-04 08:00:00,1,1857.0,2065.627778,-208.627778,9.552205,True,False,True
1,22919,2018-02-04 18:00:00,2018-02-04 19:00:00,1,1720.0,2321.694444,-601.694444,7.962828,True,False,True
2,22930,2018-02-05 05:00:00,2018-02-05 06:00:00,1,4846.0,3821.932222,1024.067778,8.648552,True,False,True
3,22941,2018-02-05 16:00:00,2018-02-05 17:00:00,1,5912.0,5208.138889,703.861111,9.008078,True,False,True
4,22952,2018-02-06 03:00:00,2018-02-06 04:00:00,1,850.0,870.898889,-20.898889,7.920271,True,False,True


,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
516,28584,2018-09-28 19:00:00,2018-09-28 20:00:00,1,3538.0,3257.520000,280.480000,3.605880,True,False,True
517,28595,2018-09-29 06:00:00,2018-09-29 07:00:00,1,2036.0,2200.785556,-164.785556,3.457781,True,False,True
518,28606,2018-09-29 17:00:00,2018-09-29 18:00:00,1,4569.0,4470.680000,98.320000,3.555620,True,False,True
519,28617,2018-09-30 04:00:00,2018-09-30 05:00:00,1,444.0,514.968889,-70.968889,3.468607,True,False,True
520,28628,2018-09-30 15:00:00,2018-09-30 16:00:00,1,4283.0,4347.321111,-64.321111,3.421602,True,False,True


O arquivo de métricas registra o número de previsões e o tempo médio por origem. Não compare o MAE bruto com bases de outra escala.